# orange-agent: fine-tuning a small LLM to emit tool calls

Built during an internship at Orange. The question: can a small open model, fine-tuned on a free Colab GPU, turn a plain request like *"Top up line 9 with 50 bucks"* into JSON a Java backend can execute?

```json
{"tool": "chargeLine", "parameters": {"lineId": "9", "amount": 50.0}}
```

**What's in this notebook**

1. Install dependencies and check the GPU
2. Load Gemma 4 E2B (4-bit) and attach LoRA adapters
3. Generate a small synthetic dataset
4. Tokenize and fine-tune
5. Run inference and parse the JSON

**Before you run it**

- Use a **T4 GPU** runtime (Runtime > Change runtime type).
- Run the install cell, then **restart the session**, then run everything from the top of section 2.
- All data is synthetic and generated below. Nothing here comes from real customers or Orange systems.

**Known limits:** 50 training examples, one phrasing, one tool (`chargeLine`), no held-out evaluation. This is a prototype, not a finished agent.

## 1. Setup

### Install dependencies

Unsloth patches the model for faster, lower-memory fine-tuning. The pins on `xformers` and `trl` are the versions that worked for this run.

After this cell finishes, restart the session (Runtime > Restart session, or Ctrl+M .) so Python picks up the new packages.

> **Note:** on my original run the `unsloth_zoo` import failed and I patched a few missing functions into the package by hand. I removed that hack from this version. If you hit `ImportError`s from `unsloth_zoo` on a newer or older release, reinstalling both `unsloth` and `unsloth-zoo` from GitHub with `--no-cache-dir` was what fixed it for me.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.30" "trl<0.15.0" peft accelerate bitsandbytes

### Check the GPU

Colab's free tier gives a Tesla T4 (about 15 GB). The T4 has no bfloat16 support, which is why training below uses fp16.

In [ ]:
!nvidia-smi

Mon May  4 09:42:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Load the model and add LoRA adapters

**Base model:** `unsloth/gemma-4-E2B-it-unsloth-bnb-4bit`, Gemma 4 E2B instruction tuned, quantized to 4 bits so it fits comfortably on a T4.

**LoRA** trains small adapter matrices instead of all the model weights. Here: rank 16, alpha 16, no dropout, applied to the attention (`q/k/v/o_proj`) and MLP (`gate/up/down_proj`) layers. `use_gradient_checkpointing="unsloth"` trades a little speed for much lower memory use.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Gemma 4 E2B loaded with LoRA adapters")

==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Status: Gemma 4 E2B is successfully loaded 


## 3. Synthetic training data

The agent is meant to call Java methods. Three tools are defined in the project:

| Tool | Parameters |
|---|---|
| `chargeLine` | `lineId`, `amount` |
| `checkLineStatus` | `lineId` |
| `emergencyStop` | `lineId` |

Only `chargeLine` appears in the training data below, so the model has not been trained on the other two.

Each example is 50 generated requests of the form *"Charge {amt} to line {id}"*. The target output is a one-line reasoning step, then the JSON action:

```
<|think|>
Action: chargeLine. ID: 12. Amt: 50.
<|channel|>{"tool": "chargeLine", "parameters": {"lineId": "12", "amount": 50.0}}
```

The prompt tokens (`<|turn|>`, `<|think|>`, `<|channel|>`) are written by hand here, so they may not match Gemma's official chat template exactly. Training and inference use the same hand-written format, which is what matters for this experiment.

In [ ]:
import json
import random
from datasets import Dataset

random.seed(3407)  # added for reproducibility; the original run was unseeded

dataset_samples = []
for _ in range(50):
    line_id = str(random.randint(1, 15))
    amount = random.choice([10, 20, 50, 100])
    dataset_samples.append({
        "content": f"Charge {amount} to line {line_id}",
        "thought": f"Action: chargeLine. ID: {line_id}. Amt: {amount}.",
        "action": {"tool": "chargeLine",
                   "parameters": {"lineId": line_id, "amount": float(amount)}},
    })

def format_chat(samples):
    return [{"text": (
        f"<bos><|turn|>system\nAgent.<|turn|>user\n{s['content']}"
        f"<|turn|>assistant\n<|think|>\n{s['thought']}\n"
        f"<|channel|>{json.dumps(s['action'])}<eos>"
    )} for s in samples]

dataset = Dataset.from_list(format_chat(dataset_samples))
print(dataset[0]["text"])

## 4. Tokenize and train

**Tokenizing:** examples are padded or truncated to 128 tokens. The pad token is set to the end-of-sequence token. Unsloth wraps the tokenizer, so `tokenizer_only` pulls out the underlying Hugging Face tokenizer.

**Training:** the plain Hugging Face `Trainer`, not Unsloth's trainer, because the Unsloth chat-template route failed for me on this setup.

| Setting | Value |
|---|---|
| Batch size | 1, with 8 gradient accumulation steps (effective batch of 8) |
| Steps | 60 (about 10 passes over 50 examples) |
| Learning rate | 2e-4 |
| Optimizer | `adamw_8bit` |
| Precision | fp16 |

Watch the loss in the logs. On a dataset this small it should fall quickly.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

tokenizer_only = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
tokenizer_only.pad_token = tokenizer_only.eos_token

def tokenize(element):
    return tokenizer_only(element["text"], truncation=True,
                          max_length=128, padding="max_length")

tokenized_dataset = dataset.map(tokenize, batched=True,
                                remove_columns=dataset.column_names)

In [ ]:
trainer = Trainer(
    model = model,
    train_dataset = tokenized_dataset,
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer_only, mlm=False),
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = True,
        optim = "adamw_8bit",
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        output_dir = "orange_agent_v1",
        report_to = "none",
        logging_steps = 1,
    ),
)

trainer.train()

## 5. Inference

Free-form prompting gave rambling output from this model, not clean JSON. I went through several fixes while debugging:

1. Greedy decoding plus a stop string
2. Priming the answer with `Action:`
3. A raw `USER: ... AGENT_ACTION:` format with a repetition penalty
4. Pre-filling the response up to the opening `{` of the JSON
5. Adding a format hint and a one-shot example to the system prompt

The version below combines the last two. It pre-fills `Action: chargeLine.` and the opening brace, so the model only has to produce the line ID and amount, then stops at the closing `}`.

**Limitation:** because the tool name is pre-filled, the model is not choosing the tool. Removing that pre-fill is the main next step (see the README).

In [ ]:
FastLanguageModel.for_inference(model)

SPEC_HINT = 'Format: {"tool": "chargeLine", "parameters": {"lineId": "ID", "amount": VAL}}'

def orange_agent(user_input):
    prompt = (
        f"<bos><|turn|>system\nAgent. {SPEC_HINT}<|turn|>user\n{user_input}"
        f"<|turn|>assistant\n<|think|>\nAction: chargeLine.\n<|channel|>{{"
    )
    inputs = tokenizer_only(prompt, return_tensors="pt",
                            add_special_tokens=False).to("cuda")
    outputs = model.generate(
        input_ids = inputs.input_ids,
        attention_mask = inputs.attention_mask,
        max_new_tokens = 64,
        do_sample = False,          # greedy decoding
        stop_strings = ["}"],       # stop when the arguments object closes
        tokenizer = tokenizer_only,
    )
    generated = tokenizer_only.batch_decode(outputs)[0][len(prompt):]
    return "{" + generated

### Try it

Edit the prompts to test your own. The first one is phrased differently from anything in the training data, so it's a quick check of how well the model handles new wording.

In [ ]:
test_prompts = [
    "Top up line 9 with 50 bucks.",
    "Add 250 units to line 12 immediately.",
    "Charge 20 to line 3",
]

for p in test_prompts:
    print(f"INPUT:  {p}")
    print(f"OUTPUT: {orange_agent(p)}\n")

### Parse the output

The backend needs valid JSON, so the last step is to parse and sanity check the result before passing it on. Anything that fails here should be rejected, not executed.

In [ ]:
def parse_action(raw):
    try:
        action = json.loads(raw)
        params = action["parameters"]
        assert action["tool"] == "chargeLine"
        assert isinstance(params["lineId"], str)
        assert float(params["amount"]) > 0
        return action
    except (json.JSONDecodeError, KeyError, AssertionError, ValueError, TypeError):
        return None

for p in test_prompts:
    print(p, "->", parse_action(orange_agent(p)))

## What's next

- Train on all three tools, plus vague and invalid requests the agent should refuse or ask about.
- Hold out a test set and report exact-match accuracy on tool name and arguments.
- Remove the pre-filled `Action: chargeLine.` so the model picks the tool itself.
- Use constrained decoding against a JSON schema.
- Save the LoRA adapter (`model.save_pretrained(...)`) or export a merged model, and call it from a mock Java service.